# conserved-field-compression — reproducible benchmarks

**Runtime → Run all.** Self-contained: installs dependencies, writes every
module, and reproduces the headline numbers.

- **ZFP+**: ZFP's ratio and pointwise error, with total energy made exact.
- **Standalone codec**: attention-routed block-local quantizer (#3).
- **Dead-ends** (downsampling, auto thresholds, rotation) live in the modules
  behind flags; see the README.

In [ ]:
!pip -q install zstandard zfpy matplotlib
import numpy, scipy, zstandard, zfpy
print("ready")

## Modules

In [ ]:
%%writefile attention.py
"""
Front-end gradient pass: builds a normalized [0,1] "attention map" over a
scientific float array, highlighting structurally important regions
(shock fronts, vortices, sharp gradients) vs. smooth/laminar background.

Supports 2D scalar fields, 2D vector fields (u, v), and 3D vector fields
(u, v, w) via curl/vorticity magnitude. Falls back to gradient-magnitude
for scalar fields.
"""
from __future__ import annotations
import numpy as np


def _minmax_normalize(x: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    lo, hi = np.min(x), np.max(x)
    if hi - lo < eps:
        return np.zeros_like(x, dtype=np.float32)
    return ((x - lo) / (hi - lo)).astype(np.float32)


def gradient_magnitude_map(field: np.ndarray) -> np.ndarray:
    """Scalar field -> normalized gradient-magnitude attention map."""
    grads = np.gradient(field.astype(np.float64))
    if field.ndim == 1:
        grads = [grads]
    mag = np.zeros_like(field, dtype=np.float64)
    for g in grads:
        mag += g ** 2
    mag = np.sqrt(mag)
    return _minmax_normalize(mag)


def curl_2d(u: np.ndarray, v: np.ndarray) -> np.ndarray:
    """2D scalar vorticity: omega = dv/dx - du/dy."""
    dv_dx = np.gradient(v, axis=1)
    du_dy = np.gradient(u, axis=0)
    return dv_dx - du_dy


def curl_3d(u: np.ndarray, v: np.ndarray, w: np.ndarray) -> np.ndarray:
    """3D vorticity vector omega = grad x (u,v,w), returns magnitude field."""
    du_dy = np.gradient(u, axis=1)
    du_dz = np.gradient(u, axis=2)
    dv_dx = np.gradient(v, axis=0)
    dv_dz = np.gradient(v, axis=2)
    dw_dx = np.gradient(w, axis=0)
    dw_dy = np.gradient(w, axis=1)

    omega_x = dw_dy - dv_dz
    omega_y = du_dz - dw_dx
    omega_z = dv_dx - du_dy

    mag = np.sqrt(omega_x ** 2 + omega_y ** 2 + omega_z ** 2)
    return mag


def vorticity_attention_map(*components: np.ndarray, smooth_sigma: float = 0.0) -> np.ndarray:
    """
    Build a normalized [0,1] attention map from a velocity field.

    - 1 component  -> scalar field, uses gradient magnitude.
    - 2 components -> 2D vector field (u, v), uses curl (vorticity).
    - 3 components -> 3D vector field (u, v, w), uses curl magnitude.

    smooth_sigma: optional Gaussian smoothing (via scipy) applied to the
    raw structural signal before normalization, to reduce single-voxel
    noise dominating attention allocation.
    """
    if len(components) == 1:
        raw = np.abs(np.gradient(components[0].astype(np.float64))[0]) \
            if components[0].ndim == 1 else None
        att_map = gradient_magnitude_map(components[0])
        raw_signal = None
    elif len(components) == 2:
        u, v = components
        raw_signal = np.abs(curl_2d(u.astype(np.float64), v.astype(np.float64)))
        att_map = None
    elif len(components) == 3:
        u, v, w = components
        raw_signal = curl_3d(u.astype(np.float64), v.astype(np.float64), w.astype(np.float64))
        att_map = None
    else:
        raise ValueError("Expected 1 (scalar), 2 (2D vector) or 3 (3D vector) components")

    if att_map is not None:
        return att_map

    if smooth_sigma > 0:
        from scipy.ndimage import gaussian_filter
        raw_signal = gaussian_filter(raw_signal, sigma=smooth_sigma)

    return _minmax_normalize(raw_signal)


In [ ]:
%%writefile bitpack.py
"""
Low-level arbitrary-bitwidth packing (1-32 bits per value) and the
adaptive, attention-routed quantization engine built on top of it.

Attention-map value -> tier -> bitwidth:
    high attention  (structure, shocks, vortex cores) -> wide bitwidth
                                                          (up to lossless float32)
    low attention   (smooth / laminar background)      -> narrow bitwidth (down to 4 bits)
"""
from __future__ import annotations
import numpy as np
from dataclasses import dataclass, field
from typing import List, Tuple, Optional


# --------------------------------------------------------------------------
# Generic bit packer: packs an array of unsigned ints (each < 2**bitwidth)
# into a tightly packed byte buffer, MSB-first within a 64-bit accumulator.
# --------------------------------------------------------------------------

def pack_bits(values: np.ndarray, bitwidth: int) -> bytes:
    if bitwidth <= 0:
        return b""
    values = values.astype(np.uint64)
    assert values.min() >= 0
    assert values.max() < (1 << bitwidth), "value exceeds bitwidth range"

    acc = 0
    acc_bits = 0
    out = bytearray()
    for v in values.tolist():
        acc = (acc << bitwidth) | v
        acc_bits += bitwidth
        while acc_bits >= 8:
            acc_bits -= 8
            out.append((acc >> acc_bits) & 0xFF)
    if acc_bits > 0:
        out.append((acc << (8 - acc_bits)) & 0xFF)
    return bytes(out)


def unpack_bits(buf: bytes, bitwidth: int, count: int) -> np.ndarray:
    if bitwidth <= 0 or count == 0:
        return np.zeros(count, dtype=np.uint32)

    mask = (1 << bitwidth) - 1
    acc = 0
    acc_bits = 0
    out = np.empty(count, dtype=np.uint64)
    idx = 0
    for byte in buf:
        acc = (acc << 8) | byte
        acc_bits += 8
        while acc_bits >= bitwidth and idx < count:
            acc_bits -= bitwidth
            out[idx] = (acc >> acc_bits) & mask
            idx += 1
        if idx >= count:
            break
    return out[:count].astype(np.uint32)


# Vectorized fast paths for the common power-of-two-friendly bitwidths used
# by this engine (4, 8, 16, 32) avoid the slow Python loop above.

def pack_bits_fast(values: np.ndarray, bitwidth: int) -> bytes:
    values = values.astype(np.uint32)
    if bitwidth == 8:
        return values.astype(np.uint8).tobytes()
    if bitwidth == 16:
        return values.astype(np.uint16).tobytes()
    if bitwidth == 32:
        return values.astype(np.uint32).tobytes()
    if bitwidth == 4:
        if values.size % 2 == 1:
            values = np.concatenate([values, [0]])
        hi = values[0::2].astype(np.uint8) << 4
        lo = values[1::2].astype(np.uint8) & 0x0F
        return (hi | lo).tobytes()
    # Fallback: generic (slow) path
    return pack_bits(values, bitwidth)


def unpack_bits_fast(buf: bytes, bitwidth: int, count: int) -> np.ndarray:
    if bitwidth == 8:
        return np.frombuffer(buf, dtype=np.uint8, count=count).astype(np.uint32)
    if bitwidth == 16:
        return np.frombuffer(buf, dtype=np.uint16, count=count).astype(np.uint32)
    if bitwidth == 32:
        return np.frombuffer(buf, dtype=np.uint32, count=count).astype(np.uint32)
    if bitwidth == 4:
        packed = np.frombuffer(buf, dtype=np.uint8)
        hi = (packed >> 4) & 0x0F
        lo = packed & 0x0F
        out = np.empty(hi.size + lo.size, dtype=np.uint32)
        out[0::2] = hi
        out[1::2] = lo
        return out[:count]
    return unpack_bits(buf, bitwidth, count)


# --------------------------------------------------------------------------
# Tiered quantization engine
# --------------------------------------------------------------------------

@dataclass
class TierSpec:
    lo: float          # attention lower bound (inclusive)
    hi: float           # attention upper bound (exclusive, 1.0 is inclusive on last tier)
    bitwidth: int        # 4, 8, 16, or 32 (32 == lossless float32 passthrough)


DEFAULT_TIERS: List[TierSpec] = [
    TierSpec(0.0, 0.15, 4),
    TierSpec(0.15, 0.45, 8),
    TierSpec(0.45, 0.75, 16),
    TierSpec(0.75, 1.0001, 32),
]


@dataclass
class TierBlock:
    bitwidth: int
    indices: np.ndarray       # flat indices into the original array (int64)
    vmin: float
    vmax: float
    packed: bytes


@dataclass
class QuantizedField:
    shape: Tuple[int, ...]
    tiers: List[TierBlock]
    tier_specs: List[TierSpec]


def assign_tiers(attention_map: np.ndarray, tier_specs: List[TierSpec] = None) -> np.ndarray:
    """Return an int8 array (same shape) giving the tier index for each element."""
    tier_specs = tier_specs or DEFAULT_TIERS
    tier_idx = np.zeros(attention_map.shape, dtype=np.int8)
    for i, spec in enumerate(tier_specs):
        mask = (attention_map >= spec.lo) & (attention_map < spec.hi)
        tier_idx[mask] = i
    return tier_idx


def quantize_tiered(field: np.ndarray, attention_map: np.ndarray,
                     tier_specs: List[TierSpec] = None) -> QuantizedField:
    tier_specs = tier_specs or DEFAULT_TIERS
    tier_idx = assign_tiers(attention_map, tier_specs)
    flat_field = field.astype(np.float32).ravel()
    flat_tier = tier_idx.ravel()

    blocks: List[TierBlock] = []
    for i, spec in enumerate(tier_specs):
        idx = np.nonzero(flat_tier == i)[0]
        if idx.size == 0:
            blocks.append(TierBlock(spec.bitwidth, idx, 0.0, 0.0, b""))
            continue
        vals = flat_field[idx]

        if spec.bitwidth == 32:
            # Lossless passthrough: store raw float32 bytes.
            packed = vals.tobytes()
            blocks.append(TierBlock(32, idx, float(vals.min()), float(vals.max()), packed))
            continue

        vmin, vmax = float(vals.min()), float(vals.max())
        if vmax - vmin < 1e-12:
            codes = np.zeros(vals.shape, dtype=np.uint32)
        else:
            levels = (1 << spec.bitwidth) - 1
            codes = np.round((vals - vmin) / (vmax - vmin) * levels).astype(np.uint32)
            codes = np.clip(codes, 0, levels)
        packed = pack_bits_fast(codes, spec.bitwidth)
        blocks.append(TierBlock(spec.bitwidth, idx, vmin, vmax, packed))

    return QuantizedField(shape=field.shape, tiers=blocks, tier_specs=tier_specs)


def dequantize_tiered(qf: QuantizedField) -> np.ndarray:
    flat = np.zeros(int(np.prod(qf.shape)), dtype=np.float32)
    for block in qf.tiers:
        if block.indices.size == 0:
            continue
        if block.bitwidth == 32:
            vals = np.frombuffer(block.packed, dtype=np.float32, count=block.indices.size)
            flat[block.indices] = vals
            continue
        levels = (1 << block.bitwidth) - 1
        codes = unpack_bits_fast(block.packed, block.bitwidth, block.indices.size)
        if levels == 0 or block.vmax - block.vmin < 1e-12:
            vals = np.full(block.indices.size, block.vmin, dtype=np.float32)
        else:
            vals = block.vmin + (codes.astype(np.float64) / levels) * (block.vmax - block.vmin)
        flat[block.indices] = vals.astype(np.float32)
    return flat.reshape(qf.shape)


def tier_report(qf: QuantizedField) -> str:
    total = int(np.prod(qf.shape))
    lines = ["Tier allocation:"]
    for i, block in enumerate(qf.tiers):
        spec = qf.tier_specs[i]
        n = block.indices.size
        pct = 100.0 * n / total if total else 0.0
        lines.append(
            f"  tier {i}  attn[{spec.lo:.2f},{spec.hi:.2f})  "
            f"bits={block.bitwidth:>2}  n={n:>8} ({pct:5.1f}%)  "
            f"packed_bytes={len(block.packed):>9}"
        )
    return "\n".join(lines)


In [ ]:
%%writefile constraint.py
"""
Global Constraint Correction (Physics Invariants).

Quantization of the low-attention background perturbs the field locally.
Left uncorrected, that perturbation biases *macroscopic* quantities derived
by integrating over the whole domain (total energy, mass, momentum), which
is exactly what downstream scientific analysis usually cares about.

This layer runs *after* quantization and *before* entropy coding. It
measures a global scalar metric (default: sum-of-squares, a stand-in for
total kinetic energy \\int |u|^2 dV up to a constant factor) on the original
field vs. the quantized/dequantized field, then algebraically corrects the
values in the low-attention ("correctable") tiers so the metric matches the
uncompressed baseline to floating point precision.

High-attention / near-lossless tiers are left untouched -- they already
carry the fidelity the viewer needs; only the background absorbs the
correction, exactly as physically expected for near-linear conserved
quantities.
"""
from __future__ import annotations
import numpy as np
from dataclasses import dataclass
from typing import Callable, List

from bitpack import QuantizedField, dequantize_tiered


def sum_of_squares(x: np.ndarray) -> float:
    return float(np.sum(x.astype(np.float64) ** 2))


def total_energy(x: np.ndarray, rho: float = 1.0) -> float:
    """0.5 * rho * sum(|u|^2) -- proportional to sum_of_squares."""
    return 0.5 * rho * sum_of_squares(x)


@dataclass
class ConstraintReport:
    metric_name: str
    original: float
    before_correction: float
    after_correction: float
    alpha: float
    relative_error_before: float
    relative_error_after: float


def correct_energy_conservation(
    original_field: np.ndarray,
    qf: QuantizedField,
    metric_fn: Callable[[np.ndarray], float] = sum_of_squares,
    metric_name: str = "sum_of_squares (~energy)",
    correctable_max_bitwidth: int = 16,
) -> ConstraintReport:
    """
    Rescale the quantized values in tiers with bitwidth <= correctable_max_bitwidth
    (i.e. the lossy background) by a single multiplicative factor alpha so that
    metric_fn(corrected_reconstruction) == metric_fn(original_field) exactly
    (up to float64 rounding), while leaving near-lossless (high-attention)
    tiers untouched.

    Solves analytically for a sum-of-squares-type metric:
        target = metric_fn(original)
        fixed  = contribution from untouched (high-precision) tiers
        S_low  = sum of squares of the *quantized* background values
        alpha  = sqrt( (target - fixed) / S_low )
    then multiplies every background value's dequantized level by alpha.

    This is exact for metric_fn == sum_of_squares (and anything proportional
    to it, e.g. total_energy). For a general metric_fn the same alpha is
    applied and the achieved metric is reported so the residual error is
    visible rather than silently hidden.
    """
    target = metric_fn(original_field)

    recon = dequantize_tiered(qf)
    before = metric_fn(recon)
    rel_before = abs(before - target) / (abs(target) + 1e-30)

    # Partition reconstructed values into "fixed" (high precision, untouched)
    # and "correctable" (background, will be rescaled).
    fixed_mask = np.zeros(recon.shape, dtype=bool)
    correctable_mask = np.zeros(recon.shape, dtype=bool)
    flat_fixed = fixed_mask.ravel()
    flat_correctable = correctable_mask.ravel()

    for block in qf.tiers:
        if block.indices.size == 0:
            continue
        if block.bitwidth > correctable_max_bitwidth:
            flat_fixed[block.indices] = True
        else:
            flat_correctable[block.indices] = True

    flat_recon = recon.ravel().astype(np.float64)
    fixed_vals = flat_recon[flat_fixed]
    correctable_vals = flat_recon[flat_correctable]

    fixed_contrib = float(np.sum(fixed_vals ** 2))
    s_low = float(np.sum(correctable_vals ** 2))

    remainder = target - fixed_contrib
    if s_low < 1e-30 or remainder < 0:
        # Degenerate case: nothing to scale, or target unreachable by scaling
        # alone (would require sign flips / negative energy) -- fall back to
        # alpha=1 (no-op) rather than producing an unphysical result.
        alpha = 1.0
    else:
        alpha = float(np.sqrt(remainder / s_low))

    flat_recon[flat_correctable] = correctable_vals * alpha
    corrected = flat_recon.reshape(recon.shape).astype(np.float32)

    after = metric_fn(corrected)
    rel_after = abs(after - target) / (abs(target) + 1e-30)

    return ConstraintReport(
        metric_name=metric_name,
        original=target,
        before_correction=before,
        after_correction=after,
        alpha=alpha,
        relative_error_before=rel_before,
        relative_error_after=rel_after,
    ), corrected


In [ ]:
%%writefile backend.py
"""
Backend entropy coding: serialize the multi-precision tiered bitstream into
a single container and run it through zstandard (ANS-family entropy coder),
compared against gzip/zlib and plain zstd-on-raw-float baselines.
"""
from __future__ import annotations
import struct
import zlib
import numpy as np
import zstandard as zstd

from bitpack import QuantizedField, TierBlock


MAGIC = b"VFC1"  # Viewer-First Compression, v1


def serialize_quantized_field(qf: QuantizedField) -> bytes:
    """Pack the tiered, multi-precision representation into one bitstream."""
    parts = [MAGIC]
    parts.append(struct.pack("<B", len(qf.shape)))
    for dim in qf.shape:
        parts.append(struct.pack("<Q", dim))
    parts.append(struct.pack("<B", len(qf.tiers)))

    for spec, block in zip(qf.tier_specs, qf.tiers):
        parts.append(struct.pack("<ffB", spec.lo, spec.hi, block.bitwidth))
        parts.append(struct.pack("<Qdd", block.indices.size, block.vmin, block.vmax))
        # indices: delta + varint-ish via plain uint32 (fine for prototype scale)
        idx = block.indices.astype(np.uint32).tobytes()
        parts.append(struct.pack("<Q", len(idx)))
        parts.append(idx)
        parts.append(struct.pack("<Q", len(block.packed)))
        parts.append(block.packed)

    return b"".join(parts)


def deserialize_quantized_field(buf: bytes):
    from bitpack import TierSpec  # local import to avoid cycle at module load
    off = 0
    assert buf[:4] == MAGIC
    off += 4
    ndim = buf[off]; off += 1
    shape = []
    for _ in range(ndim):
        (dim,) = struct.unpack_from("<Q", buf, off); off += 8
        shape.append(dim)
    n_tiers = buf[off]; off += 1

    tier_specs = []
    tiers = []
    for _ in range(n_tiers):
        lo, hi, bitwidth = struct.unpack_from("<ffB", buf, off); off += 9
        n, vmin, vmax = struct.unpack_from("<Qdd", buf, off); off += 24
        (idx_len,) = struct.unpack_from("<Q", buf, off); off += 8
        idx_bytes = buf[off:off + idx_len]; off += idx_len
        indices = np.frombuffer(idx_bytes, dtype=np.uint32).astype(np.int64)
        (packed_len,) = struct.unpack_from("<Q", buf, off); off += 8
        packed = buf[off:off + packed_len]; off += packed_len

        tier_specs.append(TierSpec(lo, hi, bitwidth))
        tiers.append(TierBlock(bitwidth, indices, vmin, vmax, packed))

    return QuantizedField(shape=tuple(shape), tiers=tiers, tier_specs=tier_specs)


def compress_zstd(data: bytes, level: int = 19) -> bytes:
    cctx = zstd.ZstdCompressor(level=level)
    return cctx.compress(data)


def compress_gzip(data: bytes, level: int = 9) -> bytes:
    return zlib.compress(data, level)


def evaluate_compression(original_field: np.ndarray, qf: QuantizedField,
                          zstd_level: int = 19) -> dict:
    raw_bytes = original_field.astype(np.float32).tobytes()

    vfc_stream = serialize_quantized_field(qf)
    vfc_zstd = compress_zstd(vfc_stream, level=zstd_level)

    baseline_zstd_on_raw = compress_zstd(raw_bytes, level=zstd_level)
    baseline_gzip_on_raw = compress_gzip(raw_bytes)

    return {
        "raw_bytes": len(raw_bytes),
        "vfc_stream_bytes": len(vfc_stream),
        "vfc_stream_plus_zstd_bytes": len(vfc_zstd),
        "baseline_zstd_on_raw_bytes": len(baseline_zstd_on_raw),
        "baseline_gzip_on_raw_bytes": len(baseline_gzip_on_raw),
        "ratio_vfc_vs_raw": len(raw_bytes) / len(vfc_zstd),
        "ratio_baseline_zstd_vs_raw": len(raw_bytes) / len(baseline_zstd_on_raw),
        "ratio_baseline_gzip_vs_raw": len(raw_bytes) / len(baseline_gzip_on_raw),
        "vfc_vs_baseline_zstd_improvement": len(baseline_zstd_on_raw) / len(vfc_zstd),
    }


In [ ]:
%%writefile pipeline.py
"""
End-to-end orchestration: attention map -> tiered quantization -> global
constraint correction -> entropy-coded backend, plus a synthetic 3D
turbulence-like test field (Taylor-Green vortex + multi-scale noise) used
to validate and report on the whole pipeline.
"""
from __future__ import annotations
import time
import numpy as np

from attention import vorticity_attention_map
from bitpack import quantize_tiered, dequantize_tiered, tier_report, DEFAULT_TIERS
from constraint import correct_energy_conservation, sum_of_squares
from backend import serialize_quantized_field, evaluate_compression


def make_synthetic_turbulence_field(n: int = 64, seed: int = 0):
    """
    Taylor-Green-vortex base flow (smooth, large-scale structure) plus
    layered multi-octave noise (small-scale turbulence) so the field has
    both a laminar background and sharp, chaotic high-vorticity regions --
    exactly the regime this framework targets.
    """
    rng = np.random.default_rng(seed)
    x = np.linspace(0, 2 * np.pi, n, endpoint=False)
    y = np.linspace(0, 2 * np.pi, n, endpoint=False)
    z = np.linspace(0, 2 * np.pi, n, endpoint=False)
    X, Y, Z = np.meshgrid(x, y, z, indexing="ij")

    u = np.cos(X) * np.sin(Y) * np.cos(Z)
    v = -np.sin(X) * np.cos(Y) * np.cos(Z)
    w = np.zeros_like(u)

    # Multi-octave noise to create localized turbulent structure.
    for octave in range(1, 5):
        scale = 1.0 / octave
        phase = rng.uniform(0, 2 * np.pi, size=3)
        u += scale * 0.3 * np.sin(octave * X + phase[0]) * np.cos(octave * Y)
        v += scale * 0.3 * np.cos(octave * Y + phase[1]) * np.sin(octave * Z)
        w += scale * 0.3 * np.sin(octave * Z + phase[2]) * np.cos(octave * X)

    # Sprinkle a few sharp localized "shock-like" bursts.
    for _ in range(6):
        cx, cy, cz = rng.integers(0, n, size=3)
        r = n // 12
        xx, yy, zz = np.ogrid[:n, :n, :n]
        dist2 = (xx - cx) ** 2 + (yy - cy) ** 2 + (zz - cz) ** 2
        burst = np.exp(-dist2 / (2 * r ** 2)) * rng.uniform(1.5, 3.0)
        u += burst
        v += burst * rng.uniform(-1, 1)

    return u.astype(np.float32), v.astype(np.float32), w.astype(np.float32)


def run_pipeline(u: np.ndarray, v: np.ndarray, w: np.ndarray,
                  smooth_sigma: float = 0.6, zstd_level: int = 19,
                  verbose: bool = True) -> dict:
    t0 = time.time()

    # 1. Front-end gradient pass -> attention map (vorticity magnitude)
    attention_map = vorticity_attention_map(u, v, w, smooth_sigma=smooth_sigma)
    t1 = time.time()

    # 2. Adaptive tiered quantization, routed by attention
    qf_u = quantize_tiered(u, attention_map, DEFAULT_TIERS)
    qf_v = quantize_tiered(v, attention_map, DEFAULT_TIERS)
    qf_w = quantize_tiered(w, attention_map, DEFAULT_TIERS)
    t2 = time.time()

    # 3. Global constraint correction (conserve total kinetic energy,
    #    proportional to sum of squares across all three components)
    def combined_energy(_ignored):
        return sum_of_squares(u) + sum_of_squares(v) + sum_of_squares(w)

    # Correct each component's background against a metric that reflects
    # its own share of total energy (component-wise sum of squares is a
    # valid conserved sub-quantity here since u, v, w are independent
    # arrays in this representation).
    report_u, corrected_u = correct_energy_conservation(u, qf_u)
    report_v, corrected_v = correct_energy_conservation(v, qf_v)
    report_w, corrected_w = correct_energy_conservation(w, qf_w)
    t3 = time.time()

    # 4. Backend entropy coding + evaluation against baselines
    eval_u = evaluate_compression(u, qf_u, zstd_level=zstd_level)
    eval_v = evaluate_compression(v, qf_v, zstd_level=zstd_level)
    eval_w = evaluate_compression(w, qf_w, zstd_level=zstd_level)
    t4 = time.time()

    # Error metrics (post constraint-correction) vs original
    def err_stats(orig, recon):
        diff = np.abs(orig.astype(np.float64) - recon.astype(np.float64))
        denom = np.abs(orig).max() + 1e-12
        return {
            "max_abs_error": float(diff.max()),
            "mean_abs_error": float(diff.mean()),
            "rmse": float(np.sqrt(np.mean(diff ** 2))),
            "max_rel_error_pct": float(100 * diff.max() / denom),
        }

    err_u = err_stats(u, corrected_u)
    err_v = err_stats(v, corrected_v)
    err_w = err_stats(w, corrected_w)

    total_raw = eval_u["raw_bytes"] + eval_v["raw_bytes"] + eval_w["raw_bytes"]
    total_vfc = eval_u["vfc_stream_plus_zstd_bytes"] + eval_v["vfc_stream_plus_zstd_bytes"] + eval_w["vfc_stream_plus_zstd_bytes"]
    total_base_zstd = eval_u["baseline_zstd_on_raw_bytes"] + eval_v["baseline_zstd_on_raw_bytes"] + eval_w["baseline_zstd_on_raw_bytes"]
    total_base_gzip = eval_u["baseline_gzip_on_raw_bytes"] + eval_v["baseline_gzip_on_raw_bytes"] + eval_w["baseline_gzip_on_raw_bytes"]

    result = {
        "timing_sec": {
            "attention_map": t1 - t0,
            "quantization": t2 - t1,
            "constraint_correction": t3 - t2,
            "entropy_coding_eval": t4 - t3,
            "total": t4 - t0,
        },
        "tier_report_u": tier_report(qf_u),
        "constraint_reports": {"u": report_u, "v": report_v, "w": report_w},
        "error_stats": {"u": err_u, "v": err_v, "w": err_w},
        "compression": {
            "total_raw_bytes": total_raw,
            "total_vfc_bytes": total_vfc,
            "total_baseline_zstd_bytes": total_base_zstd,
            "total_baseline_gzip_bytes": total_base_gzip,
            "overall_ratio_vfc_vs_raw": total_raw / total_vfc,
            "overall_ratio_baseline_zstd_vs_raw": total_raw / total_base_zstd,
            "overall_ratio_baseline_gzip_vs_raw": total_raw / total_base_gzip,
            "vfc_improvement_vs_zstd_baseline": total_base_zstd / total_vfc,
        },
        "attention_map_stats": {
            "mean": float(attention_map.mean()),
            "frac_high_attention_gt_0.75": float(np.mean(attention_map >= 0.75)),
            "frac_low_attention_lt_0.15": float(np.mean(attention_map < 0.15)),
        },
    }

    if verbose:
        print_report(result)

    return result


def print_report(result: dict):
    print("=" * 78)
    print("VIEWER-FIRST COMPRESSION -- PROTOTYPE RESULTS")
    print("=" * 78)

    print("\n[Attention map]")
    stats = result["attention_map_stats"]
    print(f"  mean attention                 : {stats['mean']:.4f}")
    print(f"  fraction high-attention (>=0.75): {stats['frac_high_attention_gt_0.75']*100:.2f}%")
    print(f"  fraction low-attention  (<0.15) : {stats['frac_low_attention_lt_0.15']*100:.2f}%")

    print("\n[Tier allocation (u-component, representative)]")
    print("  " + result["tier_report_u"].replace("\n", "\n  "))

    print("\n[Global constraint correction -- energy conservation]")
    for comp, rep in result["constraint_reports"].items():
        print(f"  component {comp}: alpha={rep.alpha:.6f}  "
              f"rel_error before={rep.relative_error_before:.3e}  "
              f"after={rep.relative_error_after:.3e}")

    print("\n[Reconstruction error vs. original, post-correction]")
    for comp, e in result["error_stats"].items():
        print(f"  component {comp}: max_abs={e['max_abs_error']:.5f}  "
              f"mean_abs={e['mean_abs_error']:.5f}  rmse={e['rmse']:.5f}  "
              f"max_rel={e['max_rel_error_pct']:.3f}%")

    print("\n[Compression ratios, combined u+v+w]")
    c = result["compression"]
    print(f"  raw float32 size                : {c['total_raw_bytes']:>10} bytes")
    print(f"  VFC tiered stream + zstd         : {c['total_vfc_bytes']:>10} bytes "
          f"({c['overall_ratio_vfc_vs_raw']:.2f}x vs raw)")
    print(f"  baseline: zstd on raw floats     : {c['total_baseline_zstd_bytes']:>10} bytes "
          f"({c['overall_ratio_baseline_zstd_vs_raw']:.2f}x vs raw)")
    print(f"  baseline: gzip on raw floats     : {c['total_baseline_gzip_bytes']:>10} bytes "
          f"({c['overall_ratio_baseline_gzip_vs_raw']:.2f}x vs raw)")
    print(f"  VFC improvement over zstd baseline: {c['vfc_improvement_vs_zstd_baseline']:.2f}x")

    print("\n[Timing]")
    for k, v in result["timing_sec"].items():
        print(f"  {k:<24}: {v*1000:8.2f} ms")

    print("=" * 78)


if __name__ == "__main__":
    u, v, w = make_synthetic_turbulence_field(n=64, seed=42)
    run_pipeline(u, v, w)


In [ ]:
%%writefile blocked.py
"""
Block-based Viewer-First quantizer (the #3 + #4 improvements).

Motivation: the original tiered quantizer uses ONE global min/max per tier,
so the same bitwidth has to span the whole dynamic range of a tier's voxels
-- wasteful. This version cuts the grid into small blocks and:

  #3  quantizes each voxel against its BLOCK's local min/max, and picks the
      block's bitwidth from the max attention inside that block. Small local
      range => the same bitwidth resolves much finer detail => lower error at
      equal bits (directly attacking where VFC loses to ZFP).

  #4  optionally DOWNSAMPLES the smoothest (lowest-attention) blocks: instead
      of storing B^3 codes, store the block mean (constant reconstruction).
      Smooth laminar background has tiny in-block variance, so this is nearly
      free in error but large in ratio (that background is ~1/3 of voxels).

Per-block overhead is tiny: a 1-byte mode + 1-byte bitwidth + two float32
(local min/max) per block, negligible next to the payload.
"""
from __future__ import annotations
import numpy as np
import struct
from dataclasses import dataclass
from typing import List, Tuple

from bitpack import pack_bits_fast, unpack_bits_fast, TierSpec, DEFAULT_TIERS

MODE_FULL = 0        # per-voxel codes at block bitwidth, block-local min/max
MODE_CONST = 1        # single block mean (constant reconstruction)
MODE_RAW32 = 2        # lossless float32 passthrough (highest tier)
MODE_DOWN2 = 3        # 2x-coarsened block, trilinear-upsampled on decode (#4)

MAGIC = b"VFB2"       # Viewer-First Blocked, v2 (adds trilinear downsample)


def _bitwidth_for_attention(a: float, tier_specs: List[TierSpec]) -> int:
    for spec in tier_specs:
        if spec.lo <= a < spec.hi:
            return spec.bitwidth
    return tier_specs[-1].bitwidth


def auto_tiers(attention_map: np.ndarray,
               bitwidths=(4, 8, 16, 32),
               quantiles=(0.60, 0.90, 0.99)) -> List[TierSpec]:
    """#2: set tier cut points from the attention histogram instead of
    hardcoding them. Each cut is a quantile of the actual attention values,
    so the split adapts per-dataset. len(quantiles) == len(bitwidths) - 1."""
    assert len(quantiles) == len(bitwidths) - 1
    a = attention_map.ravel()
    cuts = [0.0] + [float(np.quantile(a, q)) for q in quantiles] + [1.0001]
    # ensure strictly increasing (degenerate/flat attention -> nudge)
    for i in range(1, len(cuts)):
        if cuts[i] <= cuts[i - 1]:
            cuts[i] = cuts[i - 1] + 1e-6
    return [TierSpec(cuts[i], cuts[i + 1], bitwidths[i]) for i in range(len(bitwidths))]


def _coarsen_2x(blk: np.ndarray) -> np.ndarray:
    """Average-pool a (B,B,B) block by 2 -> (B/2,B/2,B/2). B must be even."""
    B = blk.shape[0]
    h = B // 2
    return blk.reshape(h, 2, h, 2, h, 2).mean(axis=(1, 3, 5))


def _upsample_2x(coarse: np.ndarray, B: int) -> np.ndarray:
    """Trilinear upsample (B/2)^3 -> B^3."""
    from scipy.ndimage import zoom
    factor = B / coarse.shape[0]
    up = zoom(coarse.astype(np.float64), factor, order=1, mode="nearest")
    # zoom can be off-by-one on exact factors; crop/pad to B
    if up.shape[0] != B:
        out = np.zeros((B, B, B), dtype=np.float64)
        s = tuple(slice(0, min(B, up.shape[d])) for d in range(3))
        out[s] = up[s]
        up = out
    return up.astype(np.float32)


def _pad_to_blocks(field: np.ndarray, B: int):
    pads = [(0, (B - (s % B)) % B) for s in field.shape]
    return np.pad(field, pads, mode="edge"), pads


def _block_view(padded: np.ndarray, B: int):
    """Return (nb0,nb1,nb2, B,B,B) block view for a 3D array."""
    n0, n1, n2 = padded.shape
    nb0, nb1, nb2 = n0 // B, n1 // B, n2 // B
    return (padded.reshape(nb0, B, nb1, B, nb2, B)
                  .transpose(0, 2, 4, 1, 3, 5)), (nb0, nb1, nb2)


@dataclass
class BlockedField:
    orig_shape: Tuple[int, ...]
    block_size: int
    stream: bytes          # fully serialized payload
    alpha: float           # energy-conservation scale on correctable blocks
    lowest_bitwidth: int


def quantize_blocked(field: np.ndarray, attention_map: np.ndarray,
                      tier_specs: List[TierSpec] = None,
                      block_size: int = 8,
                      downsample: str = "none",   # "none" | "const" | "tri"
                      downsample_low: bool = None,  # back-compat: True -> "const"
                      lowest_bitwidth: int = None,
                      correctable_max_bitwidth: int = 16,
                      conserve_energy: bool = True) -> BlockedField:
    tier_specs = tier_specs or DEFAULT_TIERS
    B = block_size
    if downsample_low is not None:      # back-compat with earlier bool API
        downsample = "const" if downsample_low else "none"
    if downsample == "tri" and B % 2 != 0:
        downsample = "none"            # trilinear needs even block size
    if lowest_bitwidth is None:
        lowest_bitwidth = min(s.bitwidth for s in tier_specs)
    # `lowest_bitwidth` = downsample threshold; `correctable_max_bitwidth` =
    # which lossy blocks absorb the energy correction (decode scales these).

    f = field.astype(np.float32)
    fpad, pads = _pad_to_blocks(f, B)
    apad, _ = _pad_to_blocks(attention_map.astype(np.float32), B)

    fblocks, (nb0, nb1, nb2) = _block_view(fpad, B)
    ablocks, _ = _block_view(apad, B)

    # Pass 1: build block records + accumulate energies (over the ORIGINAL,
    # unpadded voxels only, so padding never enters the invariant).
    block_records = []            # (mode, bitw, vmin, vmax, packed_or_none)
    correctable_energy = 0.0      # sum of squares of quantized correctable voxels
    fixed_energy = 0.0            # sum of squares of quantized fixed voxels
    # valid-voxel mask per block (drops edge padding)
    vv, _ = _block_view(_pad_to_blocks(np.ones_like(f), B)[0], B)

    def recon_block(mode, bitw, vmin, vmax, codes, blk_shape):
        if mode == MODE_RAW32:
            return codes.reshape(blk_shape)  # codes holds raw floats here
        if mode == MODE_CONST:
            return np.full(blk_shape, vmin, dtype=np.float32)
        if mode == MODE_DOWN2:
            h = blk_shape[0] // 2
            levels = (1 << bitw) - 1
            if levels == 0 or vmax - vmin < 1e-12:
                coarse = np.full((h, h, h), vmin, dtype=np.float32)
            else:
                coarse = (vmin + (codes.astype(np.float64) / levels) * (vmax - vmin)).reshape(h, h, h)
            return _upsample_2x(coarse.astype(np.float32), blk_shape[0])
        levels = (1 << bitw) - 1
        if levels == 0 or vmax - vmin < 1e-12:
            return np.full(blk_shape, vmin, dtype=np.float32)
        return (vmin + (codes.astype(np.float64) / levels) * (vmax - vmin)).reshape(blk_shape).astype(np.float32)

    for i in range(nb0):
        for j in range(nb1):
            for k in range(nb2):
                blk = fblocks[i, j, k]
                mask = vv[i, j, k].astype(bool)
                a_max = float(ablocks[i, j, k].max())
                bitw = _bitwidth_for_attention(a_max, tier_specs)
                vmin = float(blk.min()); vmax = float(blk.max())

                if bitw >= 32:
                    rec = blk.astype(np.float32)
                    block_records.append((MODE_RAW32, 0, vmin, vmax, blk.astype(np.float32)))
                    fixed_energy += float(np.sum((rec[mask].astype(np.float64)) ** 2))
                    continue

                if downsample != "none" and bitw <= lowest_bitwidth:
                    if downsample == "const":
                        mean = float(blk.mean())
                        block_records.append((MODE_CONST, 0, mean, mean, None))
                        rec = np.full(blk.shape, mean, dtype=np.float32)
                    else:  # "tri": 2x-coarsen, quantize coarse at block bitwidth
                        coarse = _coarsen_2x(blk)
                        cmin, cmax = float(coarse.min()), float(coarse.max())
                        if cmax - cmin < 1e-12:
                            ccodes = np.zeros(coarse.size, dtype=np.uint32)
                        else:
                            levels = (1 << bitw) - 1
                            ccodes = np.round((coarse.ravel() - cmin) / (cmax - cmin) * levels).astype(np.uint32)
                            ccodes = np.clip(ccodes, 0, levels)
                        block_records.append((MODE_DOWN2, bitw, cmin, cmax, ccodes))
                        rec = recon_block(MODE_DOWN2, bitw, cmin, cmax, ccodes, blk.shape)
                    correctable_energy += float(np.sum((rec[mask].astype(np.float64)) ** 2))
                    continue

                if vmax - vmin < 1e-12:
                    codes = np.zeros(blk.size, dtype=np.uint32)
                else:
                    levels = (1 << bitw) - 1
                    codes = np.round((blk.ravel() - vmin) / (vmax - vmin) * levels).astype(np.uint32)
                    codes = np.clip(codes, 0, levels)
                block_records.append((MODE_FULL, bitw, vmin, vmax, codes))
                rec = recon_block(MODE_FULL, bitw, vmin, vmax, codes, blk.shape)
                if bitw <= correctable_max_bitwidth:
                    correctable_energy += float(np.sum((rec[mask].astype(np.float64)) ** 2))
                else:
                    fixed_energy += float(np.sum((rec[mask].astype(np.float64)) ** 2))

    # Solve alpha so total energy matches the original (over valid voxels).
    alpha = 1.0
    if conserve_energy:
        target = float(np.sum(f.astype(np.float64) ** 2))
        remainder = target - fixed_energy
        if correctable_energy > 1e-30 and remainder > 0:
            alpha = float(np.sqrt(remainder / correctable_energy))

    # Pass 2: serialize (alpha stored in header; decode applies it).
    parts = [MAGIC]
    parts.append(struct.pack("<B", len(field.shape)))
    for s in field.shape:
        parts.append(struct.pack("<Q", s))
    parts.append(struct.pack("<B", B))
    parts.append(struct.pack("<Bf", correctable_max_bitwidth, alpha))
    parts.append(struct.pack("<QQQ", nb0, nb1, nb2))

    for (mode, bitw, vmin, vmax, payload) in block_records:
        if mode == MODE_RAW32:
            parts.append(struct.pack("<Bff", MODE_RAW32, vmin, vmax))
            parts.append(payload.tobytes())
        elif mode == MODE_CONST:
            parts.append(struct.pack("<Bff", MODE_CONST, vmin, vmax))
        elif mode == MODE_DOWN2:
            parts.append(struct.pack("<BBff", MODE_DOWN2, bitw, vmin, vmax))
            parts.append(pack_bits_fast(payload, bitw))
        else:
            parts.append(struct.pack("<BBff", MODE_FULL, bitw, vmin, vmax))
            parts.append(pack_bits_fast(payload, bitw))

    return BlockedField(orig_shape=field.shape, block_size=B,
                        stream=b"".join(parts), alpha=alpha,
                        lowest_bitwidth=lowest_bitwidth)


def dequantize_blocked(bf: BlockedField) -> np.ndarray:
    buf = bf.stream
    off = 0
    assert buf[:4] == MAGIC
    off += 4
    ndim = buf[off]; off += 1
    shape = []
    for _ in range(ndim):
        (s,) = struct.unpack_from("<Q", buf, off); off += 8
        shape.append(s)
    B = buf[off]; off += 1
    correctable_max_bitwidth, alpha = struct.unpack_from("<Bf", buf, off); off += 5
    nb0, nb1, nb2 = struct.unpack_from("<QQQ", buf, off); off += 24

    padded = np.zeros((nb0 * B, nb1 * B, nb2 * B), dtype=np.float32)
    out_blocks, _ = _block_view(padded, B)  # a view we can write into

    for i in range(nb0):
        for j in range(nb1):
            for k in range(nb2):
                mode = buf[off]
                if mode == MODE_RAW32:
                    _, vmin, vmax = struct.unpack_from("<Bff", buf, off); off += 9
                    n = B ** 3
                    vals = np.frombuffer(buf, dtype=np.float32, count=n, offset=off)
                    off += n * 4
                    out_blocks[i, j, k] = vals.reshape(B, B, B)  # fixed: no alpha
                elif mode == MODE_CONST:
                    _, mean, _m2 = struct.unpack_from("<Bff", buf, off); off += 9
                    out_blocks[i, j, k] = mean * alpha  # correctable
                elif mode == MODE_DOWN2:
                    _, bitw, vmin, vmax = struct.unpack_from("<BBff", buf, off); off += 10
                    h = B // 2
                    nc = h ** 3
                    levels = (1 << bitw) - 1
                    nbytes = _packed_len(nc, bitw)
                    packed = buf[off:off + nbytes]; off += nbytes
                    ccodes = unpack_bits_fast(packed, bitw, nc)
                    if levels == 0 or vmax - vmin < 1e-12:
                        coarse = np.full((h, h, h), vmin, dtype=np.float32)
                    else:
                        coarse = (vmin + (ccodes.astype(np.float64) / levels) * (vmax - vmin)).reshape(h, h, h).astype(np.float32)
                    up = _upsample_2x(coarse, B)
                    scale = alpha if bitw <= correctable_max_bitwidth else 1.0
                    out_blocks[i, j, k] = (up * scale).astype(np.float32)
                else:  # MODE_FULL
                    _, bitw, vmin, vmax = struct.unpack_from("<BBff", buf, off); off += 10
                    n = B ** 3
                    levels = (1 << bitw) - 1
                    nbytes = _packed_len(n, bitw)
                    packed = buf[off:off + nbytes]; off += nbytes
                    codes = unpack_bits_fast(packed, bitw, n)
                    if levels == 0 or vmax - vmin < 1e-12:
                        vals = np.full(n, vmin, dtype=np.float32)
                    else:
                        vals = vmin + (codes.astype(np.float64) / levels) * (vmax - vmin)
                    scale = alpha if bitw <= correctable_max_bitwidth else 1.0
                    out_blocks[i, j, k] = (vals * scale).reshape(B, B, B).astype(np.float32)

    # crop padding
    sl = tuple(slice(0, s) for s in shape)
    return padded[sl]


def _packed_len(n: int, bitw: int) -> int:
    if bitw == 8:
        return n
    if bitw == 16:
        return n * 2
    if bitw == 32:
        return n * 4
    if bitw == 4:
        return (n + 1) // 2
    # generic
    return (n * bitw + 7) // 8


In [ ]:
%%writefile rotated.py
"""
Per-block component-frame rotation (the "rotating frame" idea, vector-space
version).

Within each spatial block we compute the 3x3 second-moment matrix of the
velocity vectors and rotate every vector into that block's principal-axis
frame. Because velocity is locally coherent, this piles most of the block's
energy into component 0; components 1 and 2 collapse toward zero and become
highly compressible.

Key property: the rotation is ORTHOGONAL, so it preserves sum-of-squares
(energy) exactly -- it composes cleanly with the conservation guarantee and
needs no spatial resampling (unlike rotating the grid itself).

We store one rotation per block. A 3x3 is redundant for an orthonormal
matrix; a quaternion (4 floats) suffices, so overhead is ~16 bytes/block.
"""
from __future__ import annotations
import numpy as np

from blocked import _pad_to_blocks, _block_view, quantize_blocked, dequantize_blocked
from backend import compress_zstd


def _unblock(blocks: np.ndarray, B: int) -> np.ndarray:
    """Inverse of _block_view: (nb0,nb1,nb2,B,B,B) -> (nb0*B,nb1*B,nb2*B)."""
    nb0, nb1, nb2 = blocks.shape[:3]
    return (blocks.transpose(0, 3, 1, 4, 2, 5)
                  .reshape(nb0 * B, nb1 * B, nb2 * B))


def _make_proper(R):
    """Ensure each 3x3 in (N,3,3) is a proper rotation (det +1) by flipping
    the last principal axis where det < 0. Done BEFORE the vectors are
    rotated, so the stored quaternion matches the transform exactly."""
    R = R.copy()
    det = np.linalg.det(R)
    R[det < 0, :, 2] *= -1.0
    return R


def _mat_to_quat(R):
    """Batch (N,3,3) proper rotations -> quaternions (N,4), scalar-first.
    Uses scipy's numerically robust converter (handles the 180-deg case)."""
    from scipy.spatial.transform import Rotation
    q_xyzw = Rotation.from_matrix(R).as_quat()      # (N,4) x,y,z,w
    q = np.roll(q_xyzw, 1, axis=1)                   # -> w,x,y,z
    return q.astype(np.float32)


def _quat_to_mat(q):
    """Batch quaternions (N,4) scalar-first -> (N,3,3) rotations."""
    from scipy.spatial.transform import Rotation
    q_xyzw = np.roll(q.astype(np.float64), -1, axis=1)  # w,x,y,z -> x,y,z,w
    return Rotation.from_quat(q_xyzw).as_matrix()


def block_pca_rotate(u, v, w, B):
    """Rotate each block's vectors into local principal axes.
    Returns (u',v',w', quats, grid_meta)."""
    upad, _ = _pad_to_blocks(u.astype(np.float32), B)
    vpad, _ = _pad_to_blocks(v.astype(np.float32), B)
    wpad, _ = _pad_to_blocks(w.astype(np.float32), B)

    ub, (nb0, nb1, nb2) = _block_view(upad, B)
    vb, _ = _block_view(vpad, B)
    wb, _ = _block_view(wpad, B)
    Nb = nb0 * nb1 * nb2
    m = B ** 3

    U = ub.reshape(Nb, m); V = vb.reshape(Nb, m); W = wb.reshape(Nb, m)
    Vec = np.stack([U, V, W], axis=2)            # (Nb, m, 3)

    M = np.einsum('nmi,nmj->nij', Vec, Vec)       # (Nb,3,3) second-moment
    evals, evecs = np.linalg.eigh(M)              # ascending
    R = evecs[:, :, ::-1]                          # columns = principal axes, desc
    R = _make_proper(R)                            # det +1 so quat is exact

    Vrot = np.einsum('nmi,nij->nmj', Vec, R)      # (Nb, m, 3) in principal frame
    quats = _mat_to_quat(R)

    up = _unblock(Vrot[:, :, 0].reshape(nb0, nb1, nb2, B, B, B), B)
    vp = _unblock(Vrot[:, :, 1].reshape(nb0, nb1, nb2, B, B, B), B)
    wp = _unblock(Vrot[:, :, 2].reshape(nb0, nb1, nb2, B, B, B), B)

    meta = (nb0, nb1, nb2, u.shape)
    # crop rotated comps back to original shape
    sl = tuple(slice(0, s) for s in u.shape)
    return up[sl], vp[sl], wp[sl], quats, meta


def block_pca_unrotate(up, vp, wp, quats, meta, B):
    nb0, nb1, nb2, shape = meta
    R = _quat_to_mat(quats)                        # (Nb,3,3)
    upad, _ = _pad_to_blocks(up.astype(np.float32), B)
    vpad, _ = _pad_to_blocks(vp.astype(np.float32), B)
    wpad, _ = _pad_to_blocks(wp.astype(np.float32), B)
    ub, _ = _block_view(upad, B); vb, _ = _block_view(vpad, B); wb, _ = _block_view(wpad, B)
    Nb = nb0 * nb1 * nb2; m = B ** 3
    Vrot = np.stack([ub.reshape(Nb, m), vb.reshape(Nb, m), wb.reshape(Nb, m)], axis=2)
    # inverse rotation: Vec = Vrot @ R^T
    Vec = np.einsum('nmj,nij->nmi', Vrot, R)
    u = _unblock(Vec[:, :, 0].reshape(nb0, nb1, nb2, B, B, B), B)
    v = _unblock(Vec[:, :, 1].reshape(nb0, nb1, nb2, B, B, B), B)
    w = _unblock(Vec[:, :, 2].reshape(nb0, nb1, nb2, B, B, B), B)
    sl = tuple(slice(0, s) for s in shape)
    return u[sl], v[sl], w[sl]


In [ ]:
%%writefile zfp_plus.py
"""
ZFP+  --  a conservation-correction layer on top of ZFP.

The honest result from earlier experiments: ZFP beats our home-grown
quantizer on raw compression ratio, but ZFP (like SZ) only guarantees a
POINTWISE error bound -- it does not preserve integrated physical invariants.
A downstream energy budget, mass balance, or momentum audit run on ZFP-
decompressed data drifts by ~1e-3..1e-4.

ZFP+ keeps ZFP's ratio and pointwise bound and adds an exact global
invariant. After ZFP round-trips a field, we solve in closed form for the
single multiplicative factor that makes the reconstruction's total energy
(sum of squares) equal the original's, and store it (4 bytes / component).
On decode the factor is applied. Cost: negligible size, a bounded and tiny
increase in pointwise error (by the factor, ~1 +/- 3e-4), and total energy
becomes exact to float precision.

This is the defensible novelty: a compressor-agnostic, closed-form
conservation layer -- demonstrated here on ZFP, but it wraps any lossy
backend.
"""
from __future__ import annotations
import struct
import numpy as np

try:
    import zfpy
    HAVE_ZFP = True
except Exception:
    HAVE_ZFP = False

MAGIC = b"ZFP+"


def _energy(x):
    return float(np.sum(x.astype(np.float64) ** 2))


def conservation_alpha(original, recon):
    """Closed-form scale so energy(alpha*recon) == energy(original)."""
    e_r = _energy(recon)
    if e_r < 1e-30:
        return 1.0
    return float(np.sqrt(_energy(original) / e_r))


def compress(field: np.ndarray, tolerance: float, conserve: bool = True):
    """Return (payload_bytes, recon) where payload = ZFP stream + stored alpha."""
    assert HAVE_ZFP, "zfpy not available"
    f = field.astype(np.float32)
    zbytes = zfpy.compress_numpy(f, tolerance=tolerance)
    recon = zfpy.decompress_numpy(zbytes)
    alpha = conservation_alpha(f, recon) if conserve else 1.0
    corrected = (recon.astype(np.float64) * alpha).astype(np.float32)
    payload = MAGIC + struct.pack("<f", alpha) + struct.pack("<Q", len(zbytes)) + zbytes
    return payload, corrected


def decompress(payload: bytes) -> np.ndarray:
    assert payload[:4] == MAGIC
    (alpha,) = struct.unpack_from("<f", payload, 4)
    (zlen,) = struct.unpack_from("<Q", payload, 8)
    zbytes = payload[16:16 + zlen]
    recon = zfpy.decompress_numpy(zbytes)
    return (recon.astype(np.float64) * alpha).astype(np.float32)


def compress_vector(comps, tolerance, conserve=True):
    """Compress a list of component arrays; one alpha each. Returns
    (total_payload_bytes, [recon...])."""
    payloads = []
    recons = []
    for f in comps:
        p, r = compress(f, tolerance, conserve=conserve)
        payloads.append(p); recons.append(r)
    return sum(len(p) for p in payloads), recons


In [ ]:
%%writefile benchmark.py
"""
Extended benchmark harness for the Viewer-First Compression prototype.

Produces the concrete data points needed for an open-source release:
  - scaling across grid sizes (ratio + timing)
  - variance across random seeds (mean +/- std)
  - a FAIR, matched-error comparison against ZFP (zfpy), the actual
    state-of-the-art lossy scientific compressor -- not just lossless zstd.

Fairness note: VFC is lossy. Comparing it to lossless zstd flatters it.
The honest comparison tunes ZFP's tolerance so ZFP's max abs error is
close to VFC's, then compares compression ratios at that matched fidelity.
"""
from __future__ import annotations
import time
import numpy as np

from attention import vorticity_attention_map
from bitpack import quantize_tiered, dequantize_tiered, DEFAULT_TIERS
from constraint import correct_energy_conservation, sum_of_squares
from backend import serialize_quantized_field, compress_zstd, compress_gzip
from pipeline import make_synthetic_turbulence_field

try:
    import zfpy
    HAVE_ZFP = True
except Exception:
    HAVE_ZFP = False


def _err_stats(orig, recon):
    diff = np.abs(orig.astype(np.float64) - recon.astype(np.float64))
    denom = np.abs(orig).max() + 1e-12
    return {
        "max_abs": float(diff.max()),
        "rmse": float(np.sqrt(np.mean(diff ** 2))),
        "max_rel_pct": float(100 * diff.max() / denom),
    }


def vfc_compress_component(field, attention_map, zstd_level=19):
    """Full VFC on one component; returns (compressed_bytes, reconstruction, err)."""
    qf = quantize_tiered(field, attention_map, DEFAULT_TIERS)
    report, corrected = correct_energy_conservation(field, qf)
    stream = serialize_quantized_field(qf)
    comp = compress_zstd(stream, level=zstd_level)
    return len(comp), corrected, _err_stats(field, corrected), report


def zfp_compress_component(field, tolerance):
    comp = zfpy.compress_numpy(field.astype(np.float32), tolerance=tolerance)
    recon = zfpy.decompress_numpy(comp)
    return len(comp), recon, _err_stats(field, recon)


def match_zfp_tolerance(field, target_max_abs, lo=1e-6, hi=1.0, iters=18):
    """Bisect ZFP tolerance so its max abs error ~ target_max_abs."""
    if not HAVE_ZFP:
        return None
    best = hi
    for _ in range(iters):
        mid = np.sqrt(lo * hi)  # geometric bisection
        _, _, err = zfp_compress_component(field, mid)
        if err["max_abs"] > target_max_abs:
            hi = mid
        else:
            best = mid
            lo = mid
    return best


def run_one(n, seed, zstd_level=19, smooth_sigma=0.6):
    u, v, w = make_synthetic_turbulence_field(n=n, seed=seed)
    comps = {"u": u, "v": v, "w": w}

    t0 = time.time()
    att = vorticity_attention_map(u, v, w, smooth_sigma=smooth_sigma)
    t_att = time.time() - t0

    raw_total = 0
    vfc_total = 0
    zstd_total = 0
    gzip_total = 0
    zfp_total = 0
    max_abs_vfc = 0.0
    rmse_accum = []

    t0 = time.time()
    for name, f in comps.items():
        raw = f.astype(np.float32).tobytes()
        raw_total += len(raw)

        c_vfc, recon, err, _rep = vfc_compress_component(f, att, zstd_level)
        vfc_total += c_vfc
        max_abs_vfc = max(max_abs_vfc, err["max_abs"])
        rmse_accum.append(err["rmse"])

        zstd_total += len(compress_zstd(raw, level=zstd_level))
        gzip_total += len(compress_gzip(raw))
    t_vfc = time.time() - t0

    row = {
        "n": n,
        "seed": seed,
        "voxels": n ** 3,
        "raw_bytes": raw_total,
        "vfc_bytes": vfc_total,
        "zstd_bytes": zstd_total,
        "gzip_bytes": gzip_total,
        "vfc_ratio": raw_total / vfc_total,
        "zstd_ratio": raw_total / zstd_total,
        "gzip_ratio": raw_total / gzip_total,
        "vfc_max_abs": max_abs_vfc,
        "vfc_rmse": float(np.mean(rmse_accum)),
        "t_attention_s": t_att,
        "t_vfc_s": t_vfc,
    }

    # Matched-error ZFP comparison (fair, lossy-vs-lossy)
    if HAVE_ZFP:
        zfp_err_accum = []
        for name, f in comps.items():
            tol = match_zfp_tolerance(f, max_abs_vfc)
            c_zfp, recon_zfp, err_zfp = zfp_compress_component(f, tol)
            zfp_total += c_zfp
            zfp_err_accum.append(err_zfp["max_abs"])
        row["zfp_bytes"] = zfp_total
        row["zfp_ratio"] = raw_total / zfp_total
        row["zfp_max_abs_matched"] = float(np.mean(zfp_err_accum))
        row["vfc_vs_zfp"] = zfp_total / vfc_total  # >1 means VFC smaller

    return row


def run_suite(grid_sizes=(32, 48, 64), seeds=(0, 1, 2), zstd_level=19, verbose=True):
    rows = []
    for n in grid_sizes:
        for s in seeds:
            row = run_one(n, s, zstd_level=zstd_level)
            rows.append(row)
            if verbose:
                extra = ""
                if "zfp_ratio" in row:
                    extra = (f"  zfp={row['zfp_ratio']:.2f}x"
                             f"  vfc/zfp={row['vfc_vs_zfp']:.2f}x")
                print(f"n={n:>3} seed={s}  vfc={row['vfc_ratio']:.2f}x  "
                      f"zstd={row['zstd_ratio']:.2f}x{extra}  "
                      f"vfc_maxabs={row['vfc_max_abs']:.4f}  "
                      f"t={row['t_vfc_s']*1000:.0f}ms")
    return rows


def summarize(rows):
    import statistics as st
    by_n = {}
    for r in rows:
        by_n.setdefault(r["n"], []).append(r)
    print("\n" + "=" * 78)
    print("SUMMARY (mean over seeds)")
    print("=" * 78)
    header = f"{'grid':>6} {'vfc_ratio':>10} {'zstd_ratio':>10}"
    if HAVE_ZFP:
        header += f" {'zfp_ratio':>10} {'vfc/zfp':>8}"
    header += f" {'vfc_maxabs':>11} {'t_vfc_ms':>9}"
    print(header)
    for n, rs in sorted(by_n.items()):
        vfc = st.mean(r["vfc_ratio"] for r in rs)
        zstd = st.mean(r["zstd_ratio"] for r in rs)
        maxabs = st.mean(r["vfc_max_abs"] for r in rs)
        t = st.mean(r["t_vfc_s"] for r in rs) * 1000
        line = f"{n:>6} {vfc:>10.3f} {zstd:>10.3f}"
        if HAVE_ZFP:
            zfp = st.mean(r["zfp_ratio"] for r in rs)
            vz = st.mean(r["vfc_vs_zfp"] for r in rs)
            line += f" {zfp:>10.3f} {vz:>8.3f}"
        line += f" {maxabs:>11.5f} {t:>9.1f}"
        print(line)


if __name__ == "__main__":
    rows = run_suite(grid_sizes=(32, 48), seeds=(0, 1))
    summarize(rows)


In [ ]:
%%writefile compare_blocked.py
"""
Head-to-head: original tiered VFC vs blocked VFC (#3) vs blocked+downsample
(#4) vs ZFP, all at matched-error-tuned ZFP for a fair lossy comparison.

For each method we report: compression ratio (after zstd), max abs error,
and energy conservation drift. ZFP is tuned per method to that method's own
max error, so the ratio comparison is apples-to-apples on fidelity.
"""
from __future__ import annotations
import numpy as np

from attention import vorticity_attention_map
from bitpack import quantize_tiered, dequantize_tiered, DEFAULT_TIERS
from constraint import correct_energy_conservation
from backend import serialize_quantized_field, compress_zstd
from blocked import quantize_blocked, dequantize_blocked
from pipeline import make_synthetic_turbulence_field

try:
    import zfpy
    HAVE_ZFP = True
except Exception:
    HAVE_ZFP = False


def _err(orig, rec):
    d = np.abs(orig.astype(np.float64) - rec.astype(np.float64))
    return float(d.max())


def _energy_drift(orig, rec):
    t = float(np.sum(orig.astype(np.float64) ** 2))
    r = float(np.sum(rec.astype(np.float64) ** 2))
    return abs(r - t) / (abs(t) + 1e-30)


def tiered(field, att, level=19):
    qf = quantize_tiered(field, att, DEFAULT_TIERS)
    _, corrected = correct_energy_conservation(field, qf)
    comp = compress_zstd(serialize_quantized_field(qf), level=level)
    return len(comp), corrected


def blocked(field, att, B, ds, level=19):
    bf = quantize_blocked(field, att, block_size=B, downsample_low=ds)
    rec = dequantize_blocked(bf)
    comp = compress_zstd(bf.stream, level=level)
    return len(comp), rec


def zfp_matched(field, target_max_abs, lo=1e-7, hi=2.0, iters=22):
    lo0, hi0 = lo, hi
    for _ in range(iters):
        mid = np.sqrt(lo * hi)
        d = zfpy.decompress_numpy(zfpy.compress_numpy(field.astype(np.float32), tolerance=mid))
        if np.abs(field - d).max() > target_max_abs:
            hi = mid
        else:
            lo = mid
    tol = np.sqrt(lo * hi)
    comp = zfpy.compress_numpy(field.astype(np.float32), tolerance=tol)
    rec = zfpy.decompress_numpy(comp)
    return len(comp), rec


def run(n=64, seed=0, level=19):
    u, v, w = make_synthetic_turbulence_field(n=n, seed=seed)
    att = vorticity_attention_map(u, v, w, smooth_sigma=0.6)
    comps = {"u": u, "v": v, "w": w}
    raw = sum(f.nbytes for f in comps.values())

    methods = {
        "tiered (orig)":    lambda f: tiered(f, att, level),
        "blocked B8":       lambda f: blocked(f, att, 8, False, level),
        "blocked B4":       lambda f: blocked(f, att, 4, False, level),
        "blocked B4 +ds":   lambda f: blocked(f, att, 4, True, level),
    }

    print(f"\n{'='*86}\nn={n} seed={seed}   raw={raw} bytes\n{'='*86}")
    header = f"{'method':<18}{'ratio':>9}{'max_err':>11}{'E-drift':>11}"
    if HAVE_ZFP:
        header += f"{'ZFP ratio':>11}{'VFC/ZFP':>9}"
    print(header)

    results = {}
    for name, fn in methods.items():
        tot = 0; maxerr = 0.0; recs = {}
        for cn, f in comps.items():
            nbytes, rec = fn(f)
            tot += nbytes
            maxerr = max(maxerr, _err(f, rec))
            recs[cn] = rec
        ratio = raw / tot
        edrift = _energy_drift(
            np.stack(list(comps.values())), np.stack([recs[c] for c in comps]))
        line = f"{name:<18}{ratio:>9.2f}{maxerr:>11.4f}{edrift:>11.1e}"

        if HAVE_ZFP:
            ztot = 0
            for cn, f in comps.items():
                znbytes, _ = zfp_matched(f, maxerr)
                ztot += znbytes
            zratio = raw / ztot
            line += f"{zratio:>11.2f}{ratio/zratio:>9.2f}"
        print(line)
        results[name] = {"ratio": ratio, "max_err": maxerr, "e_drift": edrift}

    return results


if __name__ == "__main__":
    for n in (32, 64):
        run(n=n, seed=0)


In [ ]:
%%writefile conservation_test.py
"""
The comparison VFC should actually win: downstream conservation error.

ZFP/SZ bound pointwise error but do not re-close a global physical
invariant. This measures, for VFC vs ZFP at MATCHED pointwise error, how
far each one's reconstruction drifts on integrated quantities that
downstream science checks: total energy, total momentum (per component),
and mean (mass proxy).

If VFC has a reason to exist, it shows up here: near-zero conservation
drift where ZFP has nonzero drift at the same fidelity and similar-or-worse
size.
"""
from __future__ import annotations
import numpy as np

from attention import vorticity_attention_map
from bitpack import quantize_tiered, DEFAULT_TIERS
from constraint import correct_energy_conservation
from pipeline import make_synthetic_turbulence_field

try:
    import zfpy
    HAVE_ZFP = True
except Exception:
    HAVE_ZFP = False


def invariants(u, v, w):
    return {
        "energy": float(np.sum(u.astype(np.float64) ** 2
                               + v.astype(np.float64) ** 2
                               + w.astype(np.float64) ** 2)),
        "momentum_u": float(np.sum(u.astype(np.float64))),
        "momentum_v": float(np.sum(v.astype(np.float64))),
        "momentum_w": float(np.sum(w.astype(np.float64))),
    }


def rel_drift(base, other, near_zero=1.0):
    """Relative drift, but flagged None where the baseline is ~0 (relative
    error is meaningless on a near-zero integrated quantity, e.g. net
    momentum of a symmetric field)."""
    out = {}
    for k in base:
        scale = abs(base[k])
        if scale < near_zero:
            out[k] = None  # near-zero baseline: relative drift not meaningful
        else:
            out[k] = abs(other[k] - base[k]) / scale
    return out


def vfc_reconstruct(u, v, w, att):
    outs = []
    for f in (u, v, w):
        qf = quantize_tiered(f, att, DEFAULT_TIERS)
        _, corrected = correct_energy_conservation(f, qf)
        outs.append(corrected)
    return outs


def zfp_reconstruct(u, v, w, tol):
    return [zfpy.decompress_numpy(zfpy.compress_numpy(f.astype(np.float32), tolerance=tol))
            for f in (u, v, w)]


def match_tol(f, target_max_abs, lo=1e-6, hi=1.0, iters=14):
    for _ in range(iters):
        mid = np.sqrt(lo * hi)
        d = zfpy.decompress_numpy(zfpy.compress_numpy(f.astype(np.float32), tolerance=mid))
        if np.abs(f - d).max() > target_max_abs:
            hi = mid
        else:
            lo = mid
    return np.sqrt(lo * hi)


def run(n=64, seed=0):
    u, v, w = make_synthetic_turbulence_field(n=n, seed=seed)
    att = vorticity_attention_map(u, v, w, smooth_sigma=0.6)
    base = invariants(u, v, w)

    ur, vr, wr = vfc_reconstruct(u, v, w, att)
    vfc_inv = invariants(ur, vr, wr)
    vfc_maxabs = max(np.abs(u - ur).max(), np.abs(v - vr).max(), np.abs(w - wr).max())
    vfc_drift = rel_drift(base, vfc_inv)

    result = {"n": n, "seed": seed, "vfc_maxabs": float(vfc_maxabs),
              "vfc_drift": vfc_drift}

    if HAVE_ZFP:
        tols = [match_tol(f, vfc_maxabs) for f in (u, v, w)]
        uz = zfpy.decompress_numpy(zfpy.compress_numpy(u.astype(np.float32), tolerance=tols[0]))
        vz = zfpy.decompress_numpy(zfpy.compress_numpy(v.astype(np.float32), tolerance=tols[1]))
        wz = zfpy.decompress_numpy(zfpy.compress_numpy(w.astype(np.float32), tolerance=tols[2]))
        zfp_inv = invariants(uz, vz, wz)
        result["zfp_drift"] = rel_drift(base, zfp_inv)

    return result


def print_run(r):
    print(f"\n[Conservation drift @ matched pointwise error]  n={r['n']} seed={r['seed']}")
    print(f"  VFC max abs error: {r['vfc_maxabs']:.5f}")
    print(f"  {'invariant':<14}{'VFC rel drift':>16}{'ZFP rel drift':>16}")
    for k in r["vfc_drift"]:
        vd = r["vfc_drift"][k]
        zd = r.get("zfp_drift", {}).get(k, None)
        vs = "   n/a (~0 base)" if vd is None else f"{vd:>16.2e}"
        zs = "   n/a (~0 base)" if zd is None else f"{zd:>16.2e}"
        print(f"  {k:<14}{vs}{zs}")


if __name__ == "__main__":
    for s in (0, 1):
        print_run(run(n=48, seed=s))


## Headline: ZFP+ conservation layer (energy exact, ratio unchanged)

In [ ]:
import importlib, zfp_plus, numpy as np, statistics as st
importlib.reload(zfp_plus)
from pipeline import make_synthetic_turbulence_field
import zfpy

def edrift(o, r):
    t = sum(np.sum(x.astype(float)**2) for x in o)
    rr = sum(np.sum(x.astype(float)**2) for x in r)
    return abs(rr - t) / t
def mxe(o, r):
    return max(float(np.abs(a-b).max()) for a, b in zip(o, r))

print(f"{'ZFP tol':>8}{'ratio':>8}{'max_err':>10}{'ZFP drift':>12}{'ZFP+ drift':>12}")
for TOL in (0.02, 0.1, 0.3, 1.0):
    R=[];E=[];DZ=[];DP=[]
    for s in (0,1,2):
        u,v,w = make_synthetic_turbulence_field(n=64, seed=s); comps=[u,v,w]
        raw = sum(f.nbytes for f in comps)
        zt=0; zr=[]
        for f in comps:
            zb = zfpy.compress_numpy(f.astype(np.float32), tolerance=TOL); zt+=len(zb); zr.append(zfpy.decompress_numpy(zb))
        pt, pr = zfp_plus.compress_vector(comps, TOL, conserve=True)
        R.append(raw/pt); E.append(mxe(comps,pr)); DZ.append(edrift(comps,zr)); DP.append(edrift(comps,pr))
    print(f"{TOL:>8}{st.mean(R):>8.2f}{st.mean(E):>10.4f}{st.mean(DZ):>12.2e}{st.mean(DP):>12.2e}")

## Standalone codec: block-local quantization vs original tiered vs ZFP

In [ ]:
import compare_blocked, importlib
importlib.reload(compare_blocked)
for n in (32, 64):
    compare_blocked.run(n=n, seed=0)

## Downstream conservation drift (standalone codec vs ZFP)

In [ ]:
import conservation_test, importlib
importlib.reload(conservation_test)
for s in (0, 1):
    conservation_test.print_run(conservation_test.run(n=64, seed=s))